# 02 — Feature analysisHow much information do the features carry about the next session's direction?Spoiler: very little, and that is the honest answer rather than a bug to fix.

In [ ]:
import sys, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom stock_movement.config import load_configfrom stock_movement.dataset import build_datasetconfig = load_config("../configs/reproduction/readme_aapl_2026_07.yaml")dataset = build_dataset(config)dataset.summary()

## Correlation of each feature with the target

In [ ]:
correlations = dataset.X.corrwith(dataset.y.astype(float)).sort_values()fig, ax = plt.subplots(figsize=(9, 9))ax.barh(correlations.index, correlations.to_numpy(),        color=["seagreen" if v >= 0 else "indianred" for v in correlations])ax.axvline(0, color="black", lw=1)ax.set_title("Correlation with next-session direction")ax.set_xlabel("Pearson r")plt.tight_layout()n = len(dataset)print(f"largest |r|: {correlations.abs().max():.4f}")print(f"noise floor at n={n}: about {1.96 / np.sqrt(n):.4f}")print("Anything below that line is indistinguishable from zero.")

## Feature-to-feature correlationCollinear features destabilise a linear model's coefficients. `momentum_Nd` is*skip-a-day* momentum for exactly this reason — plain N-day momentum wouldduplicate `return_Nd`. Momentum windows are configured independently of volatilitywindows, so retuning one does not silently change the other.

In [ ]:
corr = dataset.X.corr()fig, ax = plt.subplots(figsize=(11, 9))im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)ax.set_yticks(range(len(corr)), corr.columns, fontsize=7)fig.colorbar(im, shrink=0.8)ax.set_title("Feature correlation matrix")plt.tight_layout()pairs = corr.where(~np.eye(len(corr), dtype=bool)).abs().stack().sort_values(ascending=False)print("most collinear pairs:")print(pairs.head(6).round(3).to_string())

## Do up and down sessions look different at all?

In [ ]:
interesting = ["return_1d", "volatility_20d", "close_to_sma_20", "volume_zscore_20"]fig, axes = plt.subplots(1, len(interesting), figsize=(16, 3.6))for ax, name in zip(axes, interesting):    for label, colour in ((1, "seagreen"), (0, "indianred")):        ax.hist(dataset.X.loc[dataset.y == label, name], bins=60, alpha=0.5,                color=colour, density=True, label="up" if label else "down")    ax.set_title(name, fontsize=10)    ax.legend(fontsize=8)plt.tight_layout()print("The distributions overlap almost perfectly — which is what a balanced")print("accuracy near 0.50 looks like before you fit anything.")

## Is the signal even stable over time?

In [ ]:
rolling = dataset.X["return_1d"].rolling(252).corr(dataset.y.astype(float))rolling.plot(figsize=(12, 4))plt.axhline(0, color="black", lw=1)plt.title("Rolling 1-year correlation: today's return vs next session's direction")plt.tight_layout()print("The relationship flips sign repeatedly. A model fitted on one regime")print("is fitted against the next one.")